In [2]:
import pandas as pd
import re

In [4]:
df = pd.read_csv("bank_data.csv")

# Standardize column names (adjust if needed)
df.columns = df.columns.str.lower()

# Keep only required columns
df = df[['type', 'mode', 'amount', 'narration']]

# Remove very small noise transactions
df = df[df['amount'] > 5]

# Convert text to uppercase
df['narration'] = df['narration'].astype(str).str.upper()

df.head()

,type,mode,amount,narration
0,DEBIT,CARD,100.0,GAS FILLING STATION
1,DEBIT,CARD,170.0,GAS FILLING STATION
2,DEBIT,CARD,500.0,GAS FILLING STATION
3,CREDIT,OTHERS,15.0,5188810
4,DEBIT,ATM,1000.0,ATM


In [6]:
def extract_merchant(text):
    keywords = [
        "AMZN", "AMAZON", "FLIPKART",
        "JIO", "AIRTEL",
        "GAS", "FUEL",
        "BILLDESK",
        "NETFLIX", "SPOTIFY",
        "ATM",
        "NEFT", "IMPS",
        "EURONET"
    ]
    
    for k in keywords:
        if k in text:
            return k
    
    # Payment gateways (uncertain merchants)
    if any(x in text for x in ["BHARATPE", "PAYTM", "GPAY", "PHONEPE", "UPI"]):
        return "UPI_MERCHANT"
    
    return "UNKNOWN"

df['merchant'] = df['narration'].apply(extract_merchant)
df.head()

,type,mode,amount,narration,merchant
0,DEBIT,CARD,100.0,GAS FILLING STATION,GAS
1,DEBIT,CARD,170.0,GAS FILLING STATION,GAS
2,DEBIT,CARD,500.0,GAS FILLING STATION,GAS
3,CREDIT,OTHERS,15.0,5188810,UNKNOWN
4,DEBIT,ATM,1000.0,ATM,ATM


In [ ]:
def assign_category(row):
    m = row['merchant']
    t = row['type']
    amt = row['amount']
    
    # Income
    if t == "CREDIT" and m in ["NEFT", "IMPS"]:
        return "Income"
    
    # Strong signals
    if m in ["AMZN", "AMAZON", "FLIPKART"]:
        return "Shopping"
    elif m in ["JIO", "AIRTEL"]:
        return "Bills"
    elif m in ["GAS", "FUEL"]:
        return "Transport"
    elif m == "ATM":
        return "Cash Withdrawal"
    elif m == "BILLDESK":
        return "Utilities"
    
    # Generic UPI → Neutral bucket
    elif m in ["UPI_GENERIC", "UNKNOWN"]:
        return "Local/UPI"
    
    else:
        return "Others"

df['category'] = df.apply(assign_category, axis=1)

In [9]:

training_data = df[df['type'] == "DEBIT"]

# Save
training_data.to_csv("training_data.csv", index=False)
df.to_csv("full_cleaned_data.csv", index=False)

print("Training rows:", len(training_data))
print(training_data['category'].value_counts())

Training rows: 674
category
Others             636
Bills               19
Utilities            6
Shopping             5
Transport            4
Cash Withdrawal      3
Local/UPI            1
Name: count, dtype: int64
